# CATSA 3-Class (Rest / Distress / Deep Flow)

이 노트북은 YAML 없이 한 파일에서 바로 학습합니다.

## Label Mapping (Nback 제외)
- `0`: Rest -> `Baseline`
- `1`: Distress -> `Stroop`
- `2`: DeepFlow -> `Logic`, `Sudoku`

## Multi-Branch Late Fusion (60초 윈도우)
- BVP(64Hz) 입력: `(B, 1, 3840)`
- ACC(32Hz) 입력: `(B, 3, 1920)`
- Slow group(4Hz) 입력: `(B, 4, 240)` where channels = `[EDA, TEMP, HR, HRV]`

브랜치 출력:
- BVP branch: `(B, 64, 240)` via stride `4 -> 4`
- ACC branch: `(B, 32, 240)` via stride `4 -> 2`
- Slow branch: `(B, 16, 240)` via stride `1`

Fusion:
- concat -> `(B, 112, 240)`
- permute -> `(B, 240, 112)`
- Transformer Encoder -> GAP -> FC(3)


In [1]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import find_peaks
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader

# ===================== User Config =====================
DATASET_ROOT = Path('/home/binghin2/Myproject/Dataset/CATSA')

TASK_TO_CLASS = {
    'Baseline': 0,
    'Stroop': 1,
    'Logic': 2,
    'Sudoku': 2,
}
CLASS_NAMES = ['Rest', 'Distress', 'DeepFlow']

WINDOW_SEC = 60
STRIDE_SEC_TRAIN = 10
STRIDE_SEC_EVAL = 60

# Native sampling rates
FS_BVP = 64
FS_ACC = 32
FS_SLOW = 4   # EDA/TEMP and derived HR/HRV

# 60 sec expected lengths
LEN_BVP = WINDOW_SEC * FS_BVP   # 3840
LEN_ACC = WINDOW_SEC * FS_ACC   # 1920
LEN_SLOW = WINDOW_SEC * FS_SLOW # 240

BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print('Device:', DEVICE)
print('Dataset root:', DATASET_ROOT)


Device: cuda
Dataset root: /home/binghin2/Myproject/Dataset/CATSA


In [ ]:
def read_csv_array(path: Path) -> np.ndarray:
    return pd.read_csv(path).values.astype(np.float32)


def bvp_to_hr_hrv_4hz(bvp_64: np.ndarray, fs: int = 64, out_fs: int = 4,
                     min_peak_distance_sec: float = 0.30,
                     hrv_window_sec: int = 10):
    """
    BVP(64Hz)에서 피크를 찾고 HR, HRV(IBI rolling std)를 계산한 뒤 4Hz로 보간.
    Returns: hr_4hz (T4,), hrv_4hz (T4,)
    """
    sig = np.asarray(bvp_64, dtype=np.float32).reshape(-1)
    min_dist = max(1, int(fs * min_peak_distance_sec))
    peaks, _ = find_peaks(sig, distance=min_dist)

    n4 = len(sig) // (fs // out_fs)
    t4 = np.arange(n4) / float(out_fs)

    if len(peaks) < 3:
        return np.zeros(n4, dtype=np.float32), np.zeros(n4, dtype=np.float32)

    t_peaks = peaks / float(fs)
    ibi = np.diff(t_peaks)
    ibi = np.clip(ibi, 1e-3, None)
    hr = 60.0 / ibi

    t_ibi = t_peaks[1:]
    hr_4 = np.interp(t4, t_ibi, hr).astype(np.float32)
    ibi_series = pd.Series(ibi)
    hrv = ibi_series.rolling(window=max(2, int(hrv_window_sec)), min_periods=1).std().fillna(0).values
    hrv_4 = np.interp(t4, t_ibi, hrv).astype(np.float32)

    return hr_4, hrv_4


def load_task_modalities(subject_dir: Path, task: str):
    task_dir = subject_dir / task
    paths = {
        'acc': task_dir / 'ACC.csv',
        'bvp': task_dir / 'BVP.csv',
        'eda': task_dir / 'EDA.csv',
        'temp': task_dir / 'TEMP.csv',
    }
    if not all(p.exists() for p in paths.values()):
        return None

    acc = read_csv_array(paths['acc'])
    bvp = read_csv_array(paths['bvp']).reshape(-1)
    eda = read_csv_array(paths['eda']).reshape(-1)
    temp = read_csv_array(paths['temp']).reshape(-1)

    # ACC 3축 보장
    if acc.ndim == 1:
        acc = acc.reshape(-1, 1)
    if acc.shape[1] < 3:
        acc = np.tile(acc, (1, 3))[:, :3]
    else:
        acc = acc[:, :3]

    # BVP -> HR/HRV (4Hz)
    hr_4, hrv_4 = bvp_to_hr_hrv_4hz(bvp, fs=FS_BVP, out_fs=FS_SLOW)

    # 시간축 정렬: 4Hz 기준 최소 길이
    t4_from_bvp = len(bvp) // (FS_BVP // FS_SLOW)
    t4_from_acc = len(acc) // (FS_ACC // FS_SLOW)
    t4 = min(len(eda), len(temp), len(hr_4), len(hrv_4), t4_from_bvp, t4_from_acc)
    if t4 < LEN_SLOW:
        return None

    eda = eda[:t4]
    temp = temp[:t4]
    hr_4 = hr_4[:t4]
    hrv_4 = hrv_4[:t4]

    bvp = bvp[: t4 * (FS_BVP // FS_SLOW)]
    acc = acc[: t4 * (FS_ACC // FS_SLOW)]

    slow = np.stack([eda, temp, hr_4, hrv_4], axis=1).astype(np.float32)  # (T4,4)

    return {
        'bvp': bvp.reshape(-1, 1).astype(np.float32),   # (T64,1)
        'acc': acc.astype(np.float32),                  # (T32,3)
        'slow': slow,                                   # (T4,4)
    }


def create_windows(mods: dict, label: int, stride_sec: int):
    """
    Returns list of dict with channel-first tensors ready for Conv1d:
    - bvp:  (1, 3840)
    - acc:  (3, 1920)
    - slow: (4, 240)
    """
    stride4 = stride_sec * FS_SLOW
    w4 = LEN_SLOW
    w32 = LEN_ACC
    w64 = LEN_BVP
    ratio32 = FS_ACC // FS_SLOW  # 8
    ratio64 = FS_BVP // FS_SLOW  # 16

    n4 = len(mods['slow'])
    out = []
    start4 = 0
    while start4 + w4 <= n4:
        s32 = start4 * ratio32
        s64 = start4 * ratio64

        slow_win = mods['slow'][start4:start4 + w4]           # (240,4)
        acc_win = mods['acc'][s32:s32 + w32]                  # (1920,3)
        bvp_win = mods['bvp'][s64:s64 + w64]                  # (3840,1)

        if len(slow_win) != w4 or len(acc_win) != w32 or len(bvp_win) != w64:
            break

        out.append({
            'slow': slow_win.T.copy(),  # (4,240)
            'acc': acc_win.T.copy(),    # (3,1920)
            'bvp': bvp_win.T.copy(),    # (1,3840)
            'y': int(label),
        })
        start4 += stride4

    return out


def list_subjects(root: Path):
    return sorted([p.name for p in root.glob('Sub*') if p.is_dir()])


def split_subjects(subjects, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(subjects))
    n = len(subjects)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    train_subjects = [subjects[i] for i in train_idx]
    val_subjects = [subjects[i] for i in val_idx]
    test_subjects = [subjects[i] for i in test_idx]
    return train_subjects, val_subjects, test_subjects


def build_split_windows(subjects, is_train=True):
    stride = STRIDE_SEC_TRAIN if is_train else STRIDE_SEC_EVAL
    all_windows = []

    for s in subjects:
        sdir = DATASET_ROOT / s
        for task, cls in TASK_TO_CLASS.items():
            mods = load_task_modalities(sdir, task)
            if mods is None:
                continue
            all_windows.extend(create_windows(mods, cls, stride_sec=stride))

    return all_windows


subjects = list_subjects(DATASET_ROOT)
tr_subj, va_subj, te_subj = split_subjects(subjects, seed=SEED)

print(f'Subject split -> train:{len(tr_subj)} val:{len(va_subj)} test:{len(te_subj)}')


In [ ]:
class CATSAMultiModalDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        return (
            torch.tensor(w['bvp'], dtype=torch.float32),   # (1,3840)
            torch.tensor(w['acc'], dtype=torch.float32),   # (3,1920)
            torch.tensor(w['slow'], dtype=torch.float32),  # (4,240)
            torch.tensor(w['y'], dtype=torch.long),
        )


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class MultiBranchCNNTransformer(nn.Module):
    """
    Input:
      bvp  = (B, 1, 3840)
      acc  = (B, 3, 1920)
      slow = (B, 4, 240)  [EDA, TEMP, HR, HRV]

    Branch output:
      bvp_f  = (B, 64, 240)  stride 4 -> 4
      acc_f  = (B, 32, 240)  stride 4 -> 2
      slow_f = (B, 16, 240)  stride 1

    Fusion:
      cat -> (B, 112, 240)
      permute -> (B, 240, 112)
    """
    def __init__(self, n_classes=3, d_model=112, nhead=8, nlayer=2, ff_dim=256, dropout=0.1):
        super().__init__()

        self.bvp_branch = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, stride=4, padding=4),   # 3840 -> 960
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=4, padding=4),  # 960 -> 240
            nn.BatchNorm1d(64),
            nn.GELU(),
        )

        self.acc_branch = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=9, stride=4, padding=4),   # 1920 -> 480
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4),  # 480 -> 240
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.slow_branch = nn.Sequential(
            nn.Conv1d(4, 16, kernel_size=5, stride=1, padding=2),   # 240 -> 240
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Conv1d(16, 16, kernel_size=3, stride=1, padding=1),  # 240 -> 240
            nn.BatchNorm1d(16),
            nn.GELU(),
        )

        self.posenc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=nlayer)
        self.norm = nn.LayerNorm(d_model)
        self.cls = nn.Linear(d_model, n_classes)

    def forward(self, bvp, acc, slow, return_features=False):
        bvp_f = self.bvp_branch(bvp)    # (B,64,240)
        acc_f = self.acc_branch(acc)    # (B,32,240)
        slow_f = self.slow_branch(slow) # (B,16,240)

        fused = torch.cat([bvp_f, acc_f, slow_f], dim=1)  # (B,112,240)
        seq = fused.permute(0, 2, 1).contiguous()         # (B,240,112)

        x = self.posenc(seq)
        x = self.transformer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        logits = self.cls(pooled)

        if return_features:
            return logits, seq
        return logits


# Shape sanity check
with torch.no_grad():
    m = MultiBranchCNNTransformer().to(DEVICE)
    bvp = torch.randn(2, 1, LEN_BVP, device=DEVICE)
    acc = torch.randn(2, 3, LEN_ACC, device=DEVICE)
    slow = torch.randn(2, 4, LEN_SLOW, device=DEVICE)
    logits, seq = m(bvp, acc, slow, return_features=True)
    print('Expected seq shape: (2, 240, 112)')
    print('Actual seq shape  :', tuple(seq.shape))
    print('Logits shape      :', tuple(logits.shape))
